# Araba Fiyatları (Car Prices)

🎯 Bu challenge’ın amacı, bir dataset hazırlamak ve şimdiye kadar öğrendiğiniz bazı feature selection tekniklerini uygulamaktır.

🚗 Arabalarla ilgili bir veri setiyle çalışıyoruz ve bir arabanın pahalı mı yoksa ucuz mu olduğunu tahmin etmek istiyoruz.

In [1]:
# Data manipulation
import numpy as np
import pandas as pd
# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns
# Sayısal bir özelliğin normal dağılım gösterip göstermediğini kontrol etme
from statsmodels.graphics.gofplots import qqplot


In [2]:
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Cars_dataset.csv"

❓ CSV dosyasını `df` adlı bir veri çerçevesine yükleyin.

In [4]:
df = pd.read_csv(url)
df.head()

,aspiration,enginelocation,carwidth,curbweight,enginetype,cylindernumber,stroke,peakrpm,price
0,std,front,64.1,2548,dohc,four,2.68,5000,expensive
1,std,front,64.1,2548,dohc,four,2.68,5000,expensive
2,std,front,65.5,2823,ohcv,six,3.47,5000,expensive
3,std,front,NaN,2337,ohc,four,3.40,5500,expensive
4,std,front,66.4,2824,ohc,five,3.40,5500,expensive


ℹ️ Dataset’in açıklaması [burada](https://drive.google.com/file/d/1ADSyjWfRGYqdXwCCN4PPC7PjQeMZ-ap-/view?usp=sharing ) mevcuttur. Egzersiz boyunca buna mutlaka referans verin.

## (1) Yinelenenler (Duplicates)

❓ Varsa, veri kümesinden yinelenenleri kaldırın. ❓

*Veri çerçevesini `df`* üzerine yazın.

In [7]:
df = df.drop_duplicates()
df.shape

(191, 9)

## (2)  Eksik değerler (Missing values)

❓ Eksik değerleri bulun ve bunları ya `strategy = "most frequent"` (kategorik değişkenler için) ya da `strategy = "median"` (sayısal değişkenler için) kullanarak doldurun ❓

### `carwidth`

<details>
    <summary> 💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>carwidth</code> sütununda eksik değerler birden fazla şekilde temsil edilmektedir. Bazıları <code>np.nan</code>, bazıları ise <code>*</code> olarak yer alır. Bunlar tespit edildikten sonra, eksik değerler verinin %30’undan daha azını oluşturduğu için medyan değerle doldurulabilir.
</details>

In [8]:
# '*' karakterlerini NaN ile değiştir ve sütunu sayısal tipe dönüştür
df['carwidth'] = pd.to_numeric(df['carwidth'].replace('*', np.nan), errors='coerce')

# Eksik değerleri medyan (median) ile doldur
df['carwidth'] = df['carwidth'].fillna(df['carwidth'].median())

### `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginelocation</code> kategorik bir feature olduğundan ve kategorilerin büyük çoğunluğu <code>front</code> olduğu için, en sık görülen değerle doldurun.
</details>

In [9]:
# En sık görülen (most frequent) değerle doldur
df['enginelocation'] = df['enginelocation'].fillna(df['enginelocation'].mode()[0])

🧪 **Kodunu test et**

In [10]:
from nbresult import ChallengeResult

result = ChallengeResult('missing_values',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D2-S-Data-car-prices/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 2 items

test_missing_values.py::TestMissing_values::test_carwidth PASSED         [ 50%]
test_missing_values.py::TestMissing_values::test_engine_location PASSED  [100%]

============================== 2 passed in 0.76s ===============================


💯 You can commit your code:

git add tests/missing_values.pickle

git commit -m 'Completed missing_values step'

git push origin master



## (3) Sayısal özelliklerin ölçeklendirilmesi (Scaling the numerical features)

In [11]:
# Hatırlatma olarak, DataFrame hakkında bazı bilgiler
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 191 entries, 0 to 204
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   aspiration      191 non-null    object 
 1   enginelocation  191 non-null    object 
 2   carwidth        191 non-null    float64
 3   curbweight      191 non-null    int64  
 4   enginetype      191 non-null    object 
 5   cylindernumber  191 non-null    object 
 6   stroke          191 non-null    float64
 7   peakrpm         191 non-null    int64  
 8   price           191 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 14.9+ KB


In [12]:
# Ve işte ölçeklendirmemiz gereken veri kümesinin sayısal özellikleri
numerical_features = df.select_dtypes(exclude=['object']).columns
numerical_features

Index(['carwidth', 'curbweight', 'stroke', 'peakrpm'], dtype='object')

❓ **Soru: Sayısal feature’ların ölçeklenmesi** ❓

Sayısal feature’ları aykırı değerler (outliers) ve dağılımları açısından inceleyin ve duruma göre aşağıdaki yöntemleri uygulayın:
- Robust Scaler
- Standard Scaler

Dönüştürülmüş değerlerle orijinal sütunları değiştirin.

### `peakrpm` , `carwidth` , & `stroke`

<details>
    <summary>💡 <i>İpucu</i> </summary>

    
ℹ️ <code>peakrpm</code>, <code>carwidth</code> ve <code>stroke</code> normal dağılıma sahiptir ancak aynı zamanda bazı aykırı değerler (outlier) içerir. Bu nedenle `RobustScaler()` kullanılması tavsiye edilir.
</details>

In [13]:
from sklearn.preprocessing import RobustScaler

# RobustScaler nesnesini oluşturalım
rb_scaler = RobustScaler()

# Belirtilen sütunları ölçeklendirelim ve orijinal sütunların üzerine yazalım
df[['peakrpm', 'carwidth', 'stroke']] = rb_scaler.fit_transform(df[['peakrpm', 'carwidth', 'stroke']])

### `curbweight`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>curbweight</code> normal bir dağılıma sahiptir ve aykırı değer (outlier) içermez. Bu nedenle Standard Scaler ile ölçeklenebilir.
</details>

In [14]:
from sklearn.preprocessing import StandardScaler

# StandardScaler nesnesini oluşturalım
std_scaler = StandardScaler()

# curbweight sütununu ölçeklendirelim
df[['curbweight']] = std_scaler.fit_transform(df[['curbweight']])

🧪 **Kodunu test et**

In [15]:
from nbresult import ChallengeResult

result = ChallengeResult('scaling',
                         dataset = df
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D2-S-Data-car-prices/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 4 items

test_scaling.py::TestScaling::test_carwidth PASSED                       [ 25%]
test_scaling.py::TestScaling::test_curbweight PASSED                     [ 50%]
test_scaling.py::TestScaling::test_peakrpm PASSED                        [ 75%]
test_scaling.py::TestScaling::test_stroke PASSED                         [100%]

============================== 4 passed in 0.52s ===============================


💯 You can commit your code:

git add tests/scaling.pickle

git commit -m 'Completed scaling step'

git push origin master



## (4) Kategorik özelliklerin kodlanması (Encoding the categorical features)

❓ **Soru: Kategorik değişkenlerin encode edilmesi** ❓

👇 Encode edilmesi gereken feature’ları inceleyin ve duruma göre aşağıdaki teknikleri uygulayın:

- One-hot encoding
- Manuel ordinal encoding

DataFrame içinde, orijinal feature’ları encode edilmiş versiyonlarıyla değiştirin.

### `aspiration` & `enginelocation`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>aspiration</code> ve <code>enginelocation</code> ikili (binary) kategorik feature’lardır.
</details>

In [16]:
# aspiration: 'std' -> 0, 'turbo' -> 1
df['aspiration'] = df['aspiration'].map({'std': 0, 'turbo': 1})

# enginelocation: 'front' -> 0, 'rear' -> 1
df['enginelocation'] = df['enginelocation'].map({'front': 0, 'rear': 1})

### `enginetype`

<details>
    <summary>💡 <i>İpucu</i> </summary>
    <br>
    ℹ️ <code>enginetype</code> çok kategorili (multicategorical) bir feature’dır ve One-hot encoding uygulanmalıdır.
</details>

In [17]:
# enginetype için One-Hot Encoding uygulayalım
# pd.get_dummies orijinal sütunu kaldırıp yerine yeni kategorik sütunlar ekler
df = pd.get_dummies(df, columns=['enginetype'], prefix='enginetype')

In [18]:
df.shape

(191, 15)

### `cylindernumber`

<details>
    <summary>💡 İpucu </summary>

ℹ️ <code>cylindernumber</code> sıralı (ordinal) bir feature’dır ve sayısal değerlere manuel olarak encode edilmelidir.

</details>

In [19]:
# Metin değerlerini karşılık gelen sayılara eşleyelim
cylinder_map = {
    'two': 2, 'three': 3, 'four': 4, 'five': 5, 
    'six': 6, 'eight': 8, 'twelve': 12
}
df['cylindernumber'] = df['cylindernumber'].map(cylinder_map)

❓ Artık `cylindernumber`’ı 2 ile 12 arasında sayısal bir feature’a dönüştürdüğünüze göre, bunu ölçeklendirmeniz gerekiyor ❓

<br/>

<details>
    <summary>💡 İpucu </summary>

`cylindernumber`’ın mevcut dağılımına bakın ve kendinize şu soruları sorun:
- Ölçekleme, bir feature’ın dağılımını etkiler mi?
- Bu feature’ın dağılımına göre en uygun ölçekleme yöntemi hangisidir?
</details>

In [20]:
from sklearn.preprocessing import RobustScaler

# cylindernumber sütununu ölçeklendirelim
rb_scaler = RobustScaler()
df[['cylindernumber']] = rb_scaler.fit_transform(df[['cylindernumber']])

<details>
<summary><i>Ölçekleme ve encoding işlemlerinden sonra DataFrame’inizin nasıl görünmesi gerektiğine dair bir ekran görüntüsü aşağıdadır</i></summary>
    
    
<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/02-Prepare-the-dataset/car_price_after_scaling_and_encoding.png">    

</details>

### `price`

👇 Hedef `price`ı kodlayın.

<details>
    <summary>💡 İpucu </summary>
    <br>
    ℹ️ <code>price</code> target değişkendir ve LabelEncoder ile encode edilmelidir.
</details>

In [21]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder nesnesini oluşturalım ve price sütununa uygulayalım
le = LabelEncoder()
df['price'] = le.fit_transform(df['price'])

🧪 **Kodunu test et**

In [22]:
from nbresult import ChallengeResult

result = ChallengeResult('encoding',
                         dataset = df)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D2-S-Data-car-prices/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 4 items

test_encoding.py::TestEncoding::test_aspiration PASSED                   [ 25%]
test_encoding.py::TestEncoding::test_enginelocation PASSED               [ 50%]
test_encoding.py::TestEncoding::test_enginetype PASSED                   [ 75%]
test_encoding.py::TestEncoding::test_price PASSED                        [100%]

============================== 4 passed in 0.50s ===============================


💯 You can commit your code:

git add tests/encoding.pickle

git commit -m 'Completed encoding step'

git push origin master



## (5) Temel Modelleme (Base Modelling)

👏 Veri kümesi ön işleme tabi tutuldu ve artık modele uyarlanmaya hazır. 

❓ **Soru: Bir classification modelini ilk kez değerlendirme** ❓

Ön işlenmiş bu dataset üzerinde bir `LogisticRegression` modeli için cross-validation çalıştırın ve elde edilen skoru `base_model_score` adlı değişkende saklayın.

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

# Özellikleri (X) ve hedef değişkeni (y) ayıralım
X = df.drop(columns=['price'])
y = df['price']

# Logistic Regression modelini tanımlayalım
# max_iter parametresini modelin yakınsaması (converge) için artırabiliriz
model = LogisticRegression(max_iter=1000)

# Cross-validation (çapraz doğrulama) uygulayalım (varsayılan 5 katlı/fold)
cv_results = cross_validate(model, X, y, cv=5)

# Test skorlarının ortalamasını alarak base_model_score değişkenine atayalım
base_model_score = cv_results['test_score'].mean()

print(f"Base Model Skoru: {base_model_score}")

Base Model Skoru: 0.8430499325236166


🧪 **Kodunu test et**

In [24]:
from nbresult import ChallengeResult

result = ChallengeResult('base_model',
                         score = base_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D2-S-Data-car-prices/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_base_model.py::TestBase_model::test_base_model_score PASSED         [100%]

============================== 1 passed in 0.15s ===============================


💯 You can commit your code:

git add tests/base_model.pickle

git commit -m 'Completed base_model step'

git push origin master



## (6) Özellik Seçimi  (Feature Selection (with _Permutation Importance_))

👩🏻‍🏫 Bir feature’ın target’ı tahmin etmede gerçekten önemli olup olmadığını tespit etmenin güçlü bir yolu şudur:

1. Bir model çalıştırın ve skorunu ölçün  
2. Bu feature’ı karıştırın (shuffle edin), modeli tekrar çalıştırın ve skoru tekrar ölçün  
    - Eğer performans **belirgin şekilde düşerse**, bu feature önemlidir ve **çıkarılmamalıdır**
    - Eğer performans **çok fazla düşmezse**, bu feature **elenebilir**

❓ **Sorular** ❓

1. Modele en az bilgi katkısı sağlayan feature’ları tespit etmek için feature permutation uygulayın.
2. Model performansının belirgin şekilde düşmeye başladığını fark edene kadar zayıf feature’ları dataset’ten çıkarın.
3. Elde ettiğiniz yeni güçlü feature set’i ile yeni bir modeli cross-validation ile değerlendirin ve skorunu `strong_model_score` adlı değişkende saklayın.

In [25]:
from sklearn.inspection import permutation_importance

# 1. Önce modeli mevcut tüm verilerle (X) eğitelim
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# 2. Permutation Importance hesaplayalım
# n_repeats: Güvenilirlik için işlemi 10 kez tekrarla
feature_importance = permutation_importance(model, X, y, n_repeats=10, random_state=1)

# 3. Sonuçları daha rahat görmek için bir DataFrame'e dönüştürelim
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance.importances_mean
}).sort_values(by="importance", ascending=False)

print("Özellik Önem Sıralaması:")
print(importance_df)

# 4. Zayıf özellikleri eleyerek (importance değeri 0 veya çok düşük olanlar) yeni bir set oluşturalım
# Not: Çıktıya bakarak önemi en düşük olanları manuel de seçebilirsin. 
# Genelde 0.05'in altındaki veya negatif değerleri eleriz.
threshold = 0.01 # Bu değeri çıktıdaki sonuçlara göre ayarlayabilirsin
strong_features = importance_df[importance_df['importance'] > threshold]['feature'].tolist()

X_strong = X[strong_features]

# 5. Sadece güçlü özelliklerle yeni modelin performansını ölçelim
cv_results_strong = cross_validate(LogisticRegression(max_iter=1000), X_strong, y, cv=5)
strong_model_score = cv_results_strong['test_score'].mean()

print(f"\nYeni Güçlü Model Skoru: {strong_model_score}")
print(f"Kullanılan Özellik Sayısı: {len(strong_features)} (Toplam: {X.shape[1]})")

Özellik Önem Sıralaması:
             feature  importance
3         curbweight    0.273822
2           carwidth    0.098429
5             stroke    0.029319
6            peakrpm    0.020942
11   enginetype_ohcf    0.018848
10    enginetype_ohc    0.013089
13  enginetype_rotor    0.011518
0         aspiration    0.006283
7    enginetype_dohc    0.005236
4     cylindernumber    0.004712
1     enginelocation    0.000524
8   enginetype_dohcv    0.000000
9       enginetype_l    0.000000
12   enginetype_ohcv    0.000000

Yeni Güçlü Model Skoru: 0.874493927125506
Kullanılan Özellik Sayısı: 7 (Toplam: 14)


🧪 **Kodunu test et**

In [26]:
from nbresult import ChallengeResult

result = ChallengeResult('strong_model',
                         score = strong_model_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D2-S-Data-car-prices/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_strong_model.py::TestStrong_model::test_strong_model_score PASSED   [100%]

============================== 1 passed in 0.14s ===============================


💯 You can commit your code:

git add tests/strong_model.pickle

git commit -m 'Completed strong_model step'

git push origin master



## Bonus -  Verilerinizi sınıflandırma (Stratifying your data) ⚖️

💡 Veriyi training ve testing olarak bölerken, dataset’imizdeki kategorik değişkenlerin oranına dikkat etmemiz gerekir — ister target `y`’nin sınıfları olsun ister `X` içindeki kategorik bir feature olsun.

Aşağıda bir örneğe bakalım 👇

❓ Orijinal `X` ve `y` verinizi sklearn’in `train_test_split` fonksiyonunu kullanarak training ve testing olarak ayırın; karşılaştırılabilir sonuçlar elde etmek için `random_state=1` ve `test_size=0.3` kullanın.

In [27]:
from sklearn.model_selection import train_test_split

# 1'den 10'a kadar random_state değerlerini deneyelim
for i in range(1, 11):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=i)
    
    # Test setindeki '1' (pahalı) sınıfının oranını hesaplayalım
    ratio = y_test.mean() 
    print(f"Random State {i} - Test Setindeki Pahalı Araba Oranı: {ratio:.2%}")

Random State 1 - Test Setindeki Pahalı Araba Oranı: 51.72%
Random State 2 - Test Setindeki Pahalı Araba Oranı: 56.90%
Random State 3 - Test Setindeki Pahalı Araba Oranı: 51.72%
Random State 4 - Test Setindeki Pahalı Araba Oranı: 44.83%
Random State 5 - Test Setindeki Pahalı Araba Oranı: 44.83%
Random State 6 - Test Setindeki Pahalı Araba Oranı: 53.45%
Random State 7 - Test Setindeki Pahalı Araba Oranı: 44.83%
Random State 8 - Test Setindeki Pahalı Araba Oranı: 55.17%
Random State 9 - Test Setindeki Pahalı Araba Oranı: 34.48%
Random State 10 - Test Setindeki Pahalı Araba Oranı: 55.17%


❓ Training dataset’inizde ve testing dataset’inizde `price` sınıfı **1** olan araçların oranını kontrol edin.

> _Ham `df` içinde bu orana baktığınızda, yaklaşık **%50 / %50** olması gerekir._

In [28]:
# stratify=y ekleyerek tekrar bölelim
for i in range(1, 11):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=i, stratify=y)
    
    ratio = y_test.mean()
    print(f"Stratify ile RS {i} - Test Setindeki Pahalı Araba Oranı: {ratio:.2%}")

Stratify ile RS 1 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 2 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 3 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 4 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 5 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 6 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 7 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 8 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 9 - Test Setindeki Pahalı Araba Oranı: 50.00%
Stratify ile RS 10 - Test Setindeki Pahalı Araba Oranı: 50.00%


☝️ Hâlâ yaklaşık olarak **%50 / %50** civarında olmalı.

***Peki random state’i değiştirirsek ne olur?***

❓ `random_state` değerlerini **1’den 10’a** kadar döngüye alın ve her seferinde training ve testing dataset’lerindeki `price` sınıfı **1** olan araçların oranını hesaplayın. ❓

In [29]:
from sklearn.model_selection import train_test_split

# 1'den 10'a kadar random_state değerlerini döngüye alalım
for i in range(1, 11):
    # Veriyi her seferinde farklı bir random_state ile bölelim
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=i)
    
    # price sınıfı 1 (pahalı) olan araçların oranlarını hesaplayalım
    # y değerleri 0 ve 1 olduğu için .mean() doğrudan oranı verir
    train_ratio = y_train.mean()
    test_ratio = y_test.mean()
    
    print(f"Random State {i}:")
    print(f"  Training setindeki oran: {train_ratio:.2%}")
    print(f"  Testing setindeki oran:  {test_ratio:.2%}")
    print("-" * 30)

Random State 1:
  Training setindeki oran: 50.38%
  Testing setindeki oran:  51.72%
------------------------------
Random State 2:
  Training setindeki oran: 48.12%
  Testing setindeki oran:  56.90%
------------------------------
Random State 3:
  Training setindeki oran: 50.38%
  Testing setindeki oran:  51.72%
------------------------------
Random State 4:
  Training setindeki oran: 53.38%
  Testing setindeki oran:  44.83%
------------------------------
Random State 5:
  Training setindeki oran: 53.38%
  Testing setindeki oran:  44.83%
------------------------------
Random State 6:
  Training setindeki oran: 49.62%
  Testing setindeki oran:  53.45%
------------------------------
Random State 7:
  Training setindeki oran: 53.38%
  Testing setindeki oran:  44.83%
------------------------------
Random State 8:
  Training setindeki oran: 48.87%
  Testing setindeki oran:  55.17%
------------------------------
Random State 9:
  Training setindeki oran: 57.89%
  Testing setindeki oran:  34.

Her seferinde oranların değiştiğini, hatta bazen oldukça ciddi şekilde değiştiğini gözlemleyeceksiniz 😱! Bu durum model performansını etkileyebilir.

❓ `train_test_split(random_state=1)` kullanılarak eğitilen bir Logistic Regression modelinin test skorunu,  
`random_state=9` kullanılarak eğitilen modelin test skoru ile karşılaştırın ❓

Eğitimi training data üzerinde yapmayı ve skoru testing data üzerinde hesaplamayı unutmayın.

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- Random State = 1 ---
# Veriyi bölelim
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X, y, test_size=0.3, random_state=1)

# Modeli eğitelim
model_1 = LogisticRegression(max_iter=1000)
model_1.fit(X_train_1, y_train_1)

# Test skorunu hesaplayalım
score_1 = model_1.score(X_test_1, y_test_1)

# --- Random State = 9 ---
# Veriyi farklı bir random state ile tekrar bölelim
X_train_9, X_test_9, y_train_9, y_test_9 = train_test_split(X, y, test_size=0.3, random_state=9)

# Modeli tekrar eğitelim
model_9 = LogisticRegression(max_iter=1000)
model_9.fit(X_train_9, y_train_9)

# Test skorunu hesaplayalım
score_9 = model_9.score(X_test_9, y_test_9)

# Sonuçları karşılaştıralım
print(f"Random State 1 Test Skoru: {score_1:.4f}")
print(f"Random State 9 Test Skoru: {score_9:.4f}")
print(f"Fark: {abs(score_1 - score_9):.4f}")

Random State 1 Test Skoru: 0.9310
Random State 9 Test Skoru: 0.7931
Fark: 0.1379


👀 `random_state=9` ile çok daha düşük bir skor görmelisiniz; çünkü bu test setindeki sınıf **1** araçların oranı %34.5 iken, training setinde bu oran %57.9’a, hatta orijinal dataset’te yaklaşık %50’ye yakındır.

Bu durum oldukça önemlidir; çünkü dataset’te oluşan bu **rastlantısal dengesizlik**, yalnızca model performansını düşürmekle kalmaz, aynı zamanda eğitim veya değerlendirme sırasında “gerçekliği” de bozabilir 🧐

***Peki bu sorunu nasıl çözebiliriz? Tren seti ve test seti arasında sınıfların dağılımını nasıl aynı tutabiliriz? 🔧***

🎁 Neyse ki sklearn’de, estimator (yani model) bir classifier olduğunda ve target bir sınıf olduğunda, bu durum `cross_validate` tarafından otomatik olarak ele alınır. 📚 [**sklearn.model_selection.cross_validate**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) dokümantasyonunda `cv` parametresini inceleyin.

Çözüm, aşağıdakini kullanmaktır:

> 📚 [**Stratification (Katmanlama)**](https://scikit-learn.org/stable/modules/cross_validation.html#stratification)

### Hedefin tabakalaşması (Stratification of the target)

💡 ***Stratification*** tekniğini `train_test_split` içinde de kullanabiliriz.

❓ Bu kez **1’den 10’a** kadar olan `random_state` döngüsünü tekrar çalıştırın, ancak bu sefer holdout yöntemine ***`stratify=y`*** parametresini de ekleyin. ❓

In [31]:
from sklearn.model_selection import train_test_split

# 1'den 10'a kadar random_state değerlerini döngüye alalım
for i in range(1, 11):
    # Bu sefer stratify=y parametresini ekliyoruz
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=i, stratify=y)
    
    # Oranları hesaplayalım
    train_ratio = y_train.mean()
    test_ratio = y_test.mean()
    
    print(f"Random State {i} (Stratified):")
    print(f"  Training setindeki oran: {train_ratio:.2%}")
    print(f"  Testing setindeki oran:  {test_ratio:.2%}")
    print("-" * 35)

Random State 1 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 2 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 3 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 4 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 5 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 6 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 7 (Stratified):
  Training setindeki oran: 51.13%
  Testing setindeki oran:  50.00%
-----------------------------------
Random State 8 (Stratified):
  Training setindeki oran: 51.13%
  Test

👀 Random state değişse bile, training ve testing verilerindeki sınıf oranları, orijinal `y` içindeki oranlarla aynı tutulur. İşte _stratification_ (katmanlama) tam olarak budur.

`train_test_split` fonksiyonunu `stratify` parametresiyle kullandığımızda, training ve testing verileri arasında **bir feature’ın oranlarını da koruyabiliriz**. Bu, özellikle aşağıdaki durumlarda son derece önemlidir:

- Churn tahmininde erkek ve kadın müşteri oranlarını korumak 🙋‍♂️ 🙋
- Ev fiyatlarını tahmin ederken büyük ve küçük evlerin oranlarını korumak 🏠 🏰
- Bir sonraki ürünü önerirken 1–5 arası review score dağılımını (multiclass!) korumak 🛍️
- vb.

Örneğin, bizim dataset’imizde `aspiration` feature’ının training ve testing verilerinde aynı oranda kalmasını istiyorsak, şu şekilde yazabiliriz:

`train_test_split(X, y, test_size=0.3, stratify=X.aspiration)`

---

Gördüğümüz gibi, **`cross_validate` [target değişkeni otomatik olarak stratify edebilir](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#:~:text=For%20int/None%20inputs%2C%20if%20the%20estimator%20is%20a%20classifier%20and%20y%20is%20either%20binary%20or%20multiclass%2C%20StratifiedKFold%20is%20used.)**, ancak **feature’lar için bunu yapmaz** 🤔 Bunun için biraz ekstra çalışmaya ihtiyacımız var.

Bunun için `StratifiedKFold` kullanmamız gerekiyor 🔬

### Tabakalaşma (Stratification - generalized)

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:

In [32]:
from sklearn.model_selection import StratifiedKFold

# Veriyi 5 fold’a bölecek bir stratified k-fold oluşturma
skf = StratifiedKFold(n_splits=5)
scores = []

# .split() metodu bir iterator oluşturur; 'X.aspiration' stratify edeceğimiz feature’dır
for train_indices, test_indices in skf.split(X, X.aspiration):

    # 'train_indices' ve 'test_indices', orantılı bölünmeler üreten indeks listeleridir
    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

    # modeli başlatma ve eğitme
    model = LogisticRegression()
    model.fit(X_train, y_train)

    # en sonunda 5 fold’un ortalamasını almak için skoru listeye ekleme
    scores.append(model.score(X_test, y_test))

np.array(scores).mean()

0.8585695006747638

📚 [**StratifiedKFold**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html), veriyi `K` parçaya bölerken belirli sütunlar (feature veya target) üzerinden stratification yapmamıza olanak tanır.

Bu sayede, ilgilendiğimiz kategorik feature’ların oranlarını koruyarak manuel bir cross-validation yapabiliriz — bunu ikili (binary) `aspiration` feature’ı ile deneyelim:


🏁 Tebrikler! Tüm veri setini hazırladınız, özellik seçimi yaptınız ve hatta tabakalaşma hakkında bilgi edindiniz 💪.

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!